In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Load certificate numbers from Excel
excel_file = 'abc_old_old.xlsx'
df = pd.read_excel(excel_file)
df['full_name'] = ""  # Initialize the new column to store names
cert_numbers = df['cert_num'].tolist()

# Website URL
url = 'https://sertifikat.uzbmb.uz/site/cert?type=1'

headers = {
    'accept': 'application/json, text/javascript, */*; q=0.01',
    'accept-language': 'en-US,en;q=0.9,ru;q=0.8,uz;q=0.7',
    'priority': 'u=1, i',
    'referer': 'https://sertifikat.uzbmb.uz/site/cert?type=1',
    'sec-ch-ua': '"Chromium";v="130", "Google Chrome";v="130", "Not?A_Brand";v="99"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36',
    'x-csrf-token': 'n4vN2bHfn0CabKuKrxVnESqj9m525_6KiIYrE1hYrpnox6no07vtDfIl_sD_ZTAnQeufXBq3yPDEtGR_EmviyQ==',
    'x-requested-with': 'XMLHttpRequest',
}

# Initialize session for maintaining cookies and configure connection pool size and retries
session = requests.Session()
session.headers.update(headers)

# Set the maximum pool size and retries
adapter = HTTPAdapter(
    pool_connections=70,  # Increase connection pool size
    pool_maxsize=70,      # Increase max number of connections per pool
    pool_block=True       # Block request if all connections are in use
)
session.mount('https://', adapter)

# Optional: Configure retries to handle network issues more gracefully
retry = Retry(
    total=5,                    # Retry a total of 5 times
    backoff_factor=0.1,          # Exponential backoff factor
    status_forcelist=[500, 502, 503, 504],  # Retry on these status codes
)
adapter = HTTPAdapter(max_retries=retry)
session.mount('https://', adapter)

# Define function to process a single certificate
def process_certificate(cert_number, index):
    result = {"index": index, "full_name": ""}

    print(f"Processing certificate {cert_number} ({index + 1}/{len(cert_numbers)})")

    # Load the page
    response = session.get(url)
    soup = BeautifulSoup(response.content, 'lxml')

    # Attempt to remove the CAPTCHA div
    captcha_div = soup.find("div", class_="col-md-12 mt-5")
    if captcha_div:
        captcha_div.decompose()

    # Find the certificate input field and set the certificate number
    cert_input = soup.find("input", {"id": "data-cert_number"})
    if cert_input:
        cert_input['value'] = cert_number

    # Attempt to locate the submission button instead of a form
    submit_button = soup.find("button", {"id": "save-see-form"})
    if not submit_button:
        result["full_name"] = "Submission button not found"
        return result

    # Prepare form data for submission
    form_data = {}
    for input_tag in soup.find_all("input"):
        name = input_tag.get("name")
        value = input_tag.get("value", "")
        form_data[name] = value
    form_data["Data[cert_number]"] = cert_number

    # Submit the form data
    form_action_url = url
    post_response = session.post(form_action_url, data=form_data)
    post_soup = BeautifulSoup(post_response.text, 'lxml')

    # Attempt to locate the full name
    name_element = post_soup.find("h5", class_="card-header text-center")
    if name_element:
        result["full_name"] = name_element.text.strip()
    else:
        error_message = post_soup.find("div", text="Invalid")
        if error_message:
            result["full_name"] = "Invalid"
        else:
            result["full_name"] = "Not found"

    return result


In [ ]:
# Process certificates in parallel using ThreadPoolExecutor
MAX_WORKERS = 10  # Adjust the number of workers based on your system's capabilities
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_certificate, cert_number, index) for index, cert_number in enumerate(cert_numbers)]
    
    for future in as_completed(futures):
        result = future.result()
        df.at[result["index"], 'full_name'] = result["full_name"]
        print(f"Certificate {result['index'] + 1} processed: {result['full_name']}")

# Save the updated DataFrame
df.to_excel("abc_new.xlsx", index=False)
print("Final results saved to output_with_names.xlsx.")
